In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import time

In [2]:
spark = SparkSession.builder \
    .appName("NYC Taxi Trip Analytics") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Version:", spark.version)

c:\Users\ML\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Version: 4.2.0


In [3]:
df = spark.read.parquet("data/yellow_tripdata_2024-01.parquet")

In [4]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [5]:
print("Total Rows :", df.count())
print("Total Columns :", len(df.columns))

Total Rows : 2964624
Total Columns : 19


In [6]:
df.show(10, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

In [7]:
print(df.columns)

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee']


In [8]:
df.dtypes

[('VendorID', 'int'),
 ('tpep_pickup_datetime', 'timestamp_ntz'),
 ('tpep_dropoff_datetime', 'timestamp_ntz'),
 ('passenger_count', 'bigint'),
 ('trip_distance', 'double'),
 ('RatecodeID', 'bigint'),
 ('store_and_fwd_flag', 'string'),
 ('PULocationID', 'int'),
 ('DOLocationID', 'int'),
 ('payment_type', 'bigint'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('improvement_surcharge', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('Airport_fee', 'double')]

In [9]:
df.describe().show()

+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+
|summary|          VendorID|   passenger_count|     trip_distance|       RatecodeID|store_and_fwd_flag|      PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee|
+-------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+----

In [10]:
from pyspark.sql.functions import col, when, count

missing = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing.show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|0       |0                   |0                    |140162         |0            |140162    |140162            |0           |0           |0           |0          |0    |0      |0        

In [11]:
print("Number of Records:", df.count())

Number of Records: 2964624


In [12]:
df.show(20, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

In [13]:
total_trips = df.count()
print("Total Trips:", total_trips)

Total Trips: 2964624


In [14]:
from pyspark.sql.functions import min

df.select(min("tpep_pickup_datetime").alias("Earliest Trip Date")).show()

+-------------------+
| Earliest Trip Date|
+-------------------+
|2002-12-31 22:59:39|
+-------------------+



In [15]:
from pyspark.sql.functions import max

df.select(max("tpep_pickup_datetime").alias("Latest Trip Date")).show()

+-------------------+
|   Latest Trip Date|
+-------------------+
|2024-02-01 00:01:15|
+-------------------+



In [16]:
df.select("VendorID").distinct().show()

print("Total Unique Vendors:",
      df.select("VendorID").distinct().count())


+--------+
|VendorID|
+--------+
|       1|
|       2|
|       6|
+--------+

Total Unique Vendors: 3


In [17]:
from pyspark.sql.functions import avg

df.select(avg("trip_distance").alias("Average Trip Distance")).show()

+---------------------+
|Average Trip Distance|
+---------------------+
|   3.6521691789583146|
+---------------------+



In [18]:
df.select(avg("fare_amount").alias("Average Fare")).show()

+------------------+
|      Average Fare|
+------------------+
|18.175061916791037|
+------------------+



In [19]:
df.select(max("fare_amount").alias("Maximum Fare")).show()

+------------+
|Maximum Fare|
+------------+
|      5000.0|
+------------+



In [20]:
df.select(min("fare_amount").alias("Minimum Fare")).show()

+------------+
|Minimum Fare|
+------------+
|      -899.0|
+------------+



In [21]:
df.select(avg("passenger_count").alias("Average Passenger Count")).show()

+-----------------------+
|Average Passenger Count|
+-----------------------+
|     1.3392808966805005|
+-----------------------+



In [22]:
df.select("payment_type").distinct().show()

print("Number of Payment Methods:",
      df.select("payment_type").distinct().count())


+------------+
|payment_type|
+------------+
|           1|
|           3|
|           2|
|           4|
|           0|
+------------+

Number of Payment Methods: 5


In [23]:
from pyspark.sql.functions import *

In [24]:
print("Original Number of Records:", df.count())

Original Number of Records: 2964624


In [25]:
df = df.dropDuplicates()

print("Records After Removing Duplicates:", df.count())

Records After Removing Duplicates: 2964624


In [26]:
df = df.filter(df.trip_distance > 0)

print("Records After Removing Invalid Trip Distances:", df.count())

Records After Removing Invalid Trip Distances: 2904253


In [27]:
df = df.filter(df.fare_amount >= 0)

print("Records After Removing Negative Fares:", df.count())

Records After Removing Negative Fares: 2870188


In [28]:
from pyspark.sql.functions import col, when, count

missing_values = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing_values.show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|0       |0                   |0                    |115293         |0            |115293    |115293            |0           |0           |0           |0          |0    |0      |0        

In [29]:
df = df.dropna()

print("Records After Removing Missing Values:", df.count())

Records After Removing Missing Values: 2754895


In [30]:
print("Final Number of Records:", df.count())

Final Number of Records: 2754895


In [31]:
filtered_df = df.filter(col("trip_distance") > 5)

filtered_df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:40:27|  2024-01-01 01:04:32|              2|          5.9|         1|                 N|         114|         256|           1|       30.3|  1.0|    0.5|      7.0

In [32]:
selected_df = df.select(
    "VendorID",
    "passenger_count",
    "trip_distance",
    "fare_amount"
)

selected_df.show(10)

+--------+---------------+-------------+-----------+
|VendorID|passenger_count|trip_distance|fare_amount|
+--------+---------------+-------------+-----------+
|       2|              2|         1.13|        7.9|
|       1|              3|          0.8|        6.5|
|       2|              1|         4.43|       31.0|
|       2|              1|         3.64|       17.7|
|       2|              2|          5.9|       30.3|
|       2|              2|         3.43|       24.7|
|       2|              1|         0.95|        8.6|
|       2|              2|         1.83|       10.7|
|       2|              1|         2.25|       14.2|
|       1|              2|          0.8|        7.2|
+--------+---------------+-------------+-----------+
only showing top 10 rows


In [33]:
fare_df = df.withColumn(
    "fare_with_tax",
    col("fare_amount") * 1.10
)

fare_df.select(
    "fare_amount",
    "fare_with_tax"
).show(10)

+-----------+------------------+
|fare_amount|     fare_with_tax|
+-----------+------------------+
|        7.9| 8.690000000000001|
|        6.5|              7.15|
|       31.0|              34.1|
|       17.7|19.470000000000002|
|       30.3|33.330000000000005|
|       24.7|             27.17|
|        8.6|              9.46|
|       10.7|             11.77|
|       14.2|15.620000000000001|
|        7.2| 7.920000000000001|
+-----------+------------------+
only showing top 10 rows


In [34]:
ordered_df = df.orderBy(
    col("fare_amount").desc()
)

ordered_df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-14 10:08:11|  2024-01-16 13:54:22|              1|        31.95|         1|                 N|         220|         220|           2|     2221.3|  0.0|    0.5|       0.

In [35]:
distinct_df = df.select(
    "VendorID"
).distinct()

distinct_df.show()

+--------+
|VendorID|
+--------+
|       1|
|       2|
+--------+



In [36]:
group_df = df.groupBy(
    "VendorID"
).agg(
    avg("fare_amount").alias("Average Fare")
)

group_df.show()

+--------+------------------+
|VendorID|      Average Fare|
+--------+------------------+
|       1|17.473450306546834|
|       2|18.721405626899394|
+--------+------------------+



In [38]:
from pyspark.sql.functions import when

vendor_df = df.select("VendorID").distinct()

vendor_df = vendor_df.withColumn(
    "Vendor_Name",
    when(col("VendorID") == 1, "Creative Mobile Technologies")
    .when(col("VendorID") == 2, "Curb Mobility")
    .otherwise("Unknown")
)

joined_df = df.join(vendor_df, on="VendorID", how="inner")

joined_df.select(
    "VendorID",
    "Vendor_Name",
    "fare_amount"
).show(10, truncate=False)

+--------+----------------------------+-----------+
|VendorID|Vendor_Name                 |fare_amount|
+--------+----------------------------+-----------+
|2       |Curb Mobility               |7.9        |
|1       |Creative Mobile Technologies|6.5        |
|2       |Curb Mobility               |31.0       |
|2       |Curb Mobility               |17.7       |
|2       |Curb Mobility               |30.3       |
|2       |Curb Mobility               |24.7       |
|2       |Curb Mobility               |8.6        |
|2       |Curb Mobility               |10.7       |
|2       |Curb Mobility               |14.2       |
|1       |Creative Mobile Technologies|7.2        |
+--------+----------------------------+-----------+
only showing top 10 rows


In [39]:
alias_df = df.groupBy("VendorID").agg(
    avg("trip_distance").alias("Average Distance")
)

alias_df.show()

+--------+------------------+
|VendorID|  Average Distance|
+--------+------------------+
|       1|3.1274710815392424|
|       2|3.3516586647741424|
+--------+------------------+



In [40]:
repartition_df = df.repartition(4)

print(
    "Number of Partitions:",
    repartition_df.rdd.getNumPartitions()
)

Number of Partitions: 4


In [41]:
df.createOrReplaceTempView("taxi_trips")

print("Temporary View Created Successfully!")

Temporary View Created Successfully!


In [42]:
spark.sql("""
SELECT COUNT(*) AS Total_Trips
FROM taxi_trips
""").show()

+-----------+
|Total_Trips|
+-----------+
|    2754895|
+-----------+



In [43]:
spark.sql("""
SELECT
    trip_distance,
    fare_amount,
    passenger_count
FROM taxi_trips
ORDER BY trip_distance DESC
LIMIT 10
""").show(truncate=False)

+-------------+-----------+---------------+
|trip_distance|fare_amount|passenger_count|
+-------------+-----------+---------------+
|15400.32     |28.9       |1              |
|10879.28     |70.0       |1              |
|1715.22      |70.0       |1              |
|971.8        |21.5       |1              |
|964.6        |39.5       |1              |
|277.4        |33.8       |2              |
|246.22       |8.6        |1              |
|233.25       |1616.5     |1              |
|210.82       |500.0      |1              |
|210.2        |650.0      |1              |
+-------------+-----------+---------------+



In [44]:
spark.sql("""
SELECT
    PULocationID,
    COUNT(*) AS Total_Trips
FROM taxi_trips
GROUP BY PULocationID
ORDER BY Total_Trips DESC
LIMIT 10
""").show()

+------------+-----------+
|PULocationID|Total_Trips|
+------------+-----------+
|         132|     138130|
|         237|     137093|
|         161|     136500|
|         236|     129604|
|         162|     102308|
|         186|     100737|
|         230|     100008|
|         142|      98906|
|         138|      87253|
|         239|      82391|
+------------+-----------+



In [45]:
spark.sql("""
SELECT
    payment_type,
    ROUND(AVG(fare_amount),2) AS Average_Fare
FROM taxi_trips
GROUP BY payment_type
ORDER BY payment_type
""").show()

+------------+------------+
|payment_type|Average_Fare|
+------------+------------+
|           1|       18.38|
|           2|       18.61|
|           3|       17.04|
|           4|       19.67|
+------------+------------+



In [46]:
spark.sql("""
SELECT
    HOUR(tpep_pickup_datetime) AS Pickup_Hour,
    COUNT(*) AS Total_Trips
FROM taxi_trips
GROUP BY Pickup_Hour
ORDER BY Total_Trips DESC
LIMIT 1
""").show()

+-----------+-----------+
|Pickup_Hour|Total_Trips|
+-----------+-----------+
|         18|     198037|
+-----------+-----------+



In [47]:
spark.sql("""
SELECT
    VendorID,
    trip_distance,
    fare_amount
FROM taxi_trips
WHERE trip_distance > 20
LIMIT 20
""").show(truncate=False)

+--------+-------------+-----------+
|VendorID|trip_distance|fare_amount|
+--------+-------------+-----------+
|2       |20.38        |70.0       |
|2       |51.93        |185.7      |
|1       |26.6         |122.0      |
|2       |22.42        |70.0       |
|1       |27.0         |120.3      |
|2       |24.11        |102.4      |
|1       |22.4         |70.0       |
|2       |20.58        |70.0       |
|1       |20.6         |70.0       |
|1       |22.1         |137.4      |
|2       |26.29        |99.6       |
|2       |32.24        |159.1      |
|2       |20.64        |70.0       |
|1       |20.4         |70.0       |
|2       |21.21        |70.0       |
|1       |20.6         |70.0       |
|2       |21.88        |70.0       |
|2       |21.3         |84.9       |
|2       |20.44        |70.0       |
|2       |21.29        |70.0       |
+--------+-------------+-----------+



In [48]:
spark.sql("""
SELECT
    MONTH(tpep_pickup_datetime) AS Month,
    ROUND(SUM(total_amount),2) AS Total_Revenue
FROM taxi_trips
GROUP BY Month
ORDER BY Month
""").show()

+-----+-------------+
|Month|Total_Revenue|
+-----+-------------+
|    1|7.542614842E7|
|    2|        90.47|
|   12|       235.12|
+-----+-------------+



In [49]:
spark.sql("""
SELECT
    VendorID,
    ROUND(AVG(trip_distance),2) AS Avg_Distance
FROM taxi_trips
GROUP BY VendorID
ORDER BY VendorID
""").show()

+--------+------------+
|VendorID|Avg_Distance|
+--------+------------+
|       1|        3.13|
|       2|        3.35|
+--------+------------+



In [50]:
spark.sql("""
SELECT
    fare_amount,
    trip_distance,
    passenger_count
FROM taxi_trips
ORDER BY fare_amount DESC
LIMIT 10
""").show(truncate=False)

+-----------+-------------+---------------+
|fare_amount|trip_distance|passenger_count|
+-----------+-------------+---------------+
|2221.3     |31.95        |1              |
|1616.5     |233.25       |1              |
|912.3      |142.62       |1              |
|899.0      |157.25       |1              |
|820.0      |0.21         |4              |
|761.1      |109.75       |1              |
|749.2      |122.47       |1              |
|744.3      |120.76       |1              |
|739.4      |119.46       |1              |
|700.0      |0.11         |4              |
+-----------+-------------+---------------+



In [51]:
spark.sql("""
SELECT
    VendorID,
    ROUND(AVG(passenger_count),2) AS Avg_Passengers
FROM taxi_trips
GROUP BY VendorID
ORDER BY VendorID
""").show()

+--------+--------------+
|VendorID|Avg_Passengers|
+--------+--------------+
|       1|          1.19|
|       2|          1.39|
+--------+--------------+



In [52]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank

In [53]:
window_spec = Window.orderBy(col("fare_amount").desc())

row_number_df = df.select(
    "VendorID",
    "fare_amount",
    row_number().over(window_spec).alias("Row_Number")
)

row_number_df.show(10)

+--------+-----------+----------+
|VendorID|fare_amount|Row_Number|
+--------+-----------+----------+
|       2|     2221.3|         1|
|       2|     1616.5|         2|
|       2|      912.3|         3|
|       2|      899.0|         4|
|       2|      820.0|         5|
|       2|      761.1|         6|
|       2|      749.2|         7|
|       2|      744.3|         8|
|       2|      739.4|         9|
|       2|      700.0|        10|
+--------+-----------+----------+
only showing top 10 rows


In [54]:
rank_df = df.select(
    "VendorID",
    "fare_amount",
    rank().over(window_spec).alias("Rank")
)

rank_df.show(10)

+--------+-----------+----+
|VendorID|fare_amount|Rank|
+--------+-----------+----+
|       2|     2221.3|   1|
|       2|     1616.5|   2|
|       2|      912.3|   3|
|       2|      899.0|   4|
|       2|      820.0|   5|
|       2|      761.1|   6|
|       2|      749.2|   7|
|       2|      744.3|   8|
|       2|      739.4|   9|
|       2|      700.0|  10|
+--------+-----------+----+
only showing top 10 rows


In [55]:
dense_rank_df = df.select(
    "VendorID",
    "fare_amount",
    dense_rank().over(window_spec).alias("Dense_Rank")
)

dense_rank_df.show(10)

+--------+-----------+----------+
|VendorID|fare_amount|Dense_Rank|
+--------+-----------+----------+
|       2|     2221.3|         1|
|       2|     1616.5|         2|
|       2|      912.3|         3|
|       2|      899.0|         4|
|       2|      820.0|         5|
|       2|      761.1|         6|
|       2|      749.2|         7|
|       2|      744.3|         8|
|       2|      739.4|         9|
|       2|      700.0|        10|
+--------+-----------+----------+
only showing top 10 rows


In [57]:
import time
import builtins

start_time = time.time()

df.groupBy("VendorID").count().show()

end_time = time.time()

print(
    "Execution Time Before Cache:",
    builtins.round(end_time - start_time, 2),
    "seconds"
)

+--------+-------+
|VendorID|  count|
+--------+-------+
|       1| 669555|
|       2|2085340|
+--------+-------+

Execution Time Before Cache: 40.86 seconds


In [58]:
df.cache()

# Materialize the cache
df.count()

print("Dataset Cached Successfully!")

Dataset Cached Successfully!


In [59]:
start_time = time.time()

df.groupBy("VendorID").count().show()

end_time = time.time()

print("Execution Time After Cache:",
      builtins.round(end_time - start_time, 2),
      "seconds")

+--------+-------+
|VendorID|  count|
+--------+-------+
|       1| 669555|
|       2|2085340|
+--------+-------+

Execution Time After Cache: 10.91 seconds


In [60]:
repartition_df = df.repartition(4)

print("Number of Partitions:",
      repartition_df.rdd.getNumPartitions())

Number of Partitions: 4


In [61]:
df.groupBy("VendorID").count().explain(True)

== Parsed Logical Plan ==
'Aggregate ['VendorID], ['VendorID, 'count(1) AS count#4226]
+- Filter atleastnnonnulls(19, VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18)
   +- Filter (fare_amount#10 >= cast(0 as double))
      +- Filter (trip_distance#4 > cast(0 as double))
         +- Deduplicate [DOLocationID#8, improvement_surcharge#15, tpep_dropoff_datetime#2, PULocationID#7, tolls_amount#14, tip_amount#13, passenger_count#3L, store_and_fwd_flag#6, extra#11, congestion_surcharge#17, total_amount#16, tpep_pickup_datetime#1, mta_tax#12, trip_distance#4, Airport_fee#18, RatecodeID#5L, VendorID#0, payment_type#9L, fare_amount#10]
            +- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_date

In [63]:
df.cache()
df.count()

2754895

In [64]:
query1 = spark.sql("""
SELECT
    trip_distance,
    fare_amount,
    passenger_count
FROM taxi_trips
ORDER BY trip_distance DESC
LIMIT 10
""")

query1.show()

query1.toPandas().to_csv("output/query1.csv", index=False)

+-------------+-----------+---------------+
|trip_distance|fare_amount|passenger_count|
+-------------+-----------+---------------+
|     15400.32|       28.9|              1|
|     10879.28|       70.0|              1|
|      1715.22|       70.0|              1|
|        971.8|       21.5|              1|
|        964.6|       39.5|              1|
|        277.4|       33.8|              2|
|       246.22|        8.6|              1|
|       233.25|     1616.5|              1|
|       210.82|      500.0|              1|
|        210.2|      650.0|              1|
+-------------+-----------+---------------+



c:\Users\ML\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\ML\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [65]:
query2 = spark.sql("""
SELECT
    payment_type,
    ROUND(AVG(fare_amount),2) AS Average_Fare
FROM taxi_trips
GROUP BY payment_type
""")

query2.show()

query2.toPandas().to_csv("output/query2.csv", index=False)

+------------+------------+
|payment_type|Average_Fare|
+------------+------------+
|           1|       18.38|
|           3|       17.04|
|           2|       18.61|
|           4|       19.67|
+------------+------------+



c:\Users\ML\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\ML\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [66]:
query3 = spark.sql("""
SELECT
    PULocationID,
    COUNT(*) AS Total_Trips
FROM taxi_trips
GROUP BY PULocationID
ORDER BY Total_Trips DESC
LIMIT 10
""")

query3.show()

query3.toPandas().to_csv("output/query3.csv", index=False)

+------------+-----------+
|PULocationID|Total_Trips|
+------------+-----------+
|         132|     138130|
|         237|     137093|
|         161|     136500|
|         236|     129604|
|         162|     102308|
|         186|     100737|
|         230|     100008|
|         142|      98906|
|         138|      87253|
|         239|      82391|
+------------+-----------+



c:\Users\ML\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\ML\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
